# Visual Feature Extraction (Phase 5)

Runs in Google Colab. Heavy, real-dataset feature extraction happens here,
not on a laptop. This notebook calls `evat.features.*` / `evat.tracking.*`
/ `evat.video.*` — it does not reimplement crop/encode/cache logic.

This is NOT the Video Transformer (Phase 6). It only produces per-track
temporal feature sequences for a future Transformer to consume.

Dataset: YouTube-VOS (non-commercial research use only).

In [ ]:
%pip install -q -e .

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import os
from pathlib import Path

os.environ["EVAT_DATA_ROOT"] = "/content/data"  # example only; set to the real path
dataset_root = Path(os.environ["EVAT_DATA_ROOT"]) / "youtube_vos"

## SMOKE EXPERIMENT

Verify the model loads, objects can be cropped, features are produced,
temporal sequences are constructed, the cache works, and dimensions are
correct — on a tiny subset — before any larger run.

In [ ]:
import yaml

from evat.data.datasets.youtube_vos import build_video_index
from evat.features.cache import FeatureCache, compute_cache_key, hash_config
from evat.features.encoders import CNNEncoderConfig, CNNFeatureEncoder
from evat.features.extract import extract_object_features_cnn
from evat.features.temporal import build_temporal_feature_sequence, group_features_by_track
from evat.tracking.ground_truth import extract_ground_truth_instances, strip_identity
from evat.tracking.tracker import Tracker, TrackerConfig
from evat.video.sampling import uniform_frame_indices
from evat.video.sequence import build_temporal_sequence
from evat.video.tensors import load_temporal_sequence

with open("configs/features.yaml") as f:
    feature_config = yaml.safe_load(f)

encoder_config = CNNEncoderConfig(
    backbone=feature_config["backbone"],
    pretrained=True,  # real weights, downloaded here in Colab only
    frozen=feature_config["frozen"],
    input_height=feature_config["input_height"],
    input_width=feature_config["input_width"],
    feature_dim=feature_config["feature_dim"],
)
encoder = CNNFeatureEncoder(encoder_config).to(device)
config_hash = hash_config(encoder_config)
cache = FeatureCache(feature_config["cache"]["directory"])
tracker_config = TrackerConfig.from_yaml("configs/tracking.yaml")

all_videos = build_video_index(dataset_root, split="train")
smoke_videos = all_videos[:2]


def run_feature_extraction(video, num_samples=16):
    indices = uniform_frame_indices(num_frames_total=len(video.frames), num_samples=num_samples)
    sequence = build_temporal_sequence(video, indices)
    batch = load_temporal_sequence(sequence, dataset_root=dataset_root)

    tracker = Tracker(tracker_config)
    frame_order = list(batch.frame_ids)
    all_features = []
    crop_padding = feature_config["crop"]["padding"]
    crop_mask_aware = feature_config["crop"]["mask_aware"]

    for i, frame_id in enumerate(frame_order):
        object_id_mask = batch.masks[i]
        tracked = tracker.update(
            frame_id,
            strip_identity(extract_ground_truth_instances(object_id_mask, frame_id))
            if object_id_mask is not None
            else [],
        )
        frame_rgb = batch.images[i].transpose(1, 2, 0)

        cached_this_frame = []
        to_compute = []
        for instance in tracked:
            key = compute_cache_key(
                "youtube_vos",
                video.video_id,
                frame_id,
                instance.track_id,
                encoder.name,
                config_hash,
            )
            cached = cache.get(key, expected_config_hash=config_hash)
            if cached is not None:
                cached_this_frame.append((key, instance, cached))
            else:
                to_compute.append((key, instance))

        if to_compute:
            computed = extract_object_features_cnn(
                frame_rgb,
                [inst for _, inst in to_compute],
                encoder,
                padding=crop_padding,
                mask_aware=crop_mask_aware,
            )
            for (key, _), feature in zip(to_compute, computed, strict=True):
                cache.put(key, feature.feature, config_hash=config_hash)
                all_features.append(feature)

        for _, instance, cached_vector in cached_this_frame:
            from evat.features.schemas import VisualFeature

            all_features.append(
                VisualFeature(
                    frame_id=frame_id,
                    track_id=instance.track_id,
                    feature=cached_vector,
                    extractor_name=encoder.name,
                )
            )

    return frame_order, all_features


for video in smoke_videos:
    frame_order, features = run_feature_extraction(video)
    grouped = group_features_by_track(features)
    for track_id, features_by_frame in grouped.items():
        seq = build_temporal_feature_sequence(
            track_id,
            frame_order,
            features_by_frame,
            sequence_length=feature_config["temporal"]["sequence_length"],
            stride=feature_config["temporal"]["stride"],
        )
        print(video.video_id, track_id, seq.features.shape, seq.validity.sum(), "/", seq.length)

## BASELINE EVALUATION SUBSET + THROUGHPUT

Only run after the smoke experiment succeeds. Measures real extraction
throughput; not claimed as a specific FPS/real-time number unless printed
here.

In [ ]:
import time

subset_videos = all_videos[:20]  # adjust based on observed Colab session limits

start = time.time()
total_frames = 0
total_features = 0
num_tracks = 0
for video in subset_videos:
    frame_order, features = run_feature_extraction(video)
    total_frames += len(frame_order)
    total_features += len(features)
    num_tracks += len(group_features_by_track(features))
runtime_seconds = time.time() - start

throughput = total_features / runtime_seconds if runtime_seconds > 0 else float("nan")

print("videos:", len(subset_videos))
print("frames processed:", total_frames)
print("features extracted:", total_features)
print("tracks:", num_tracks)
print("runtime_seconds:", runtime_seconds)
print("throughput (features/sec):", throughput)

## Baseline vs. learned feature comparison (optional)

If computationally practical, extract `BaselineStatsEncoder` features over
the same subset and compare — e.g. nearest-neighbor consistency of
same-track feature vectors across frames vs. different tracks. Not
performed by default in this notebook; run and document explicitly if
attempted, otherwise `docs/experiments.md` records this as not performed.

## Save results

Record the actual printed shapes/counts/runtime above into
`docs/experiments.md`. Do not hand-edit numbers not produced by this
notebook.